# 05-CPU 搭建与内部数据通路

> 把前面学到的寄存器、ALU、总线和控制信号组合起来，观察一个教学 CPU 如何在内部搬运数据并完成运算。

## 1. 本篇学习范围

本篇对应课程视频 **53—60**：

- 53_计算机系统的核心组件
- 54_构建一个8位的寄存器
- 55_构建一个4位的寄存器
- 56_ALU加法器
- 57_CPU内部原理图ALU部分
- 58_内部数据总线的绘制
- 59_CPU内部结构的完成
- 60_计算机概念梳理

学习主线是：先认识 CPU 的组件，再把寄存器和 ALU 接到内部总线上，最后用数据通路解释一条运算是怎样完成的。第 61 个视频开始进入汇编语言和指令集，留给第 06 篇。

## 2. 计算机系统的核心组件

一个能运行程序的计算机，至少需要几类相互配合的部件：

| 组件 | 主要职责 |
|---|---|
| CPU | 读取指令、控制各模块并完成运算 |
| ALU | 完成加法、减法和按位逻辑运算 |
| 寄存器 | 在 CPU 内部暂存操作数、中间结果和地址 |
| 内存 | 保存指令和数据 |
| 控制单元 | 产生读写、选择、加载和时序信号 |
| 总线 | 在组件之间传递地址、数据和控制信息 |

本篇聚焦 CPU 内部。可以先把 CPU 看成“寄存器 + ALU + 总线 + 控制信号”的组合：寄存器保存状态，ALU 处理数据，总线负责搬运数据，控制逻辑决定每一步谁读、谁写、做什么。

## 3. CPU 内部数据通路是什么

**数据通路（Datapath）**是 CPU 内部数据经过的路径和处理部件集合。它描述“数据从哪里来、经过什么运算、到哪里去”。

一个简化路径可以表示为：

```text
寄存器 A ──> ALU 输入 A
寄存器 B ──> ALU 输入 B
                 │
                 ▼
              ALU 结果 ──> 内部数据总线 ──> 目标寄存器
```

控制信号不属于数据本身，但它决定：

- 哪个寄存器把数据放到总线上；
- 哪个寄存器在时钟边沿加载总线数据；
- ALU 执行哪一种运算；
- 结果是否更新状态位。

因此，分析 CPU 电路时要把“数据线”和“控制线”分开看。

## 4. 构建 8 位寄存器

8 位寄存器可以保存一个 8 位数据，也可以保存一个字节大小的中间结果。它由 8 个 1 位存储单元并行组成，共享时钟和写入控制。

### 4.1 典型接口

| 信号 | 作用 |
|---|---|
| `D[7:0]` | 8 位输入数据 |
| `Q[7:0]` | 8 位保存结果 |
| `CLK` | 时钟边沿 |
| `LOAD` | 是否加载输入 |
| `OE` | 是否驱动外部总线 |

当 `LOAD` 有效时，寄存器在规定时钟边沿采样 `D[7:0]`；当 `LOAD` 无效时，即使输入变化，已保存的 `Q[7:0]` 仍保持不变。

### 4.2 8 位的意义

8 位宽度决定了一次能并行处理多少位数据，也决定了无符号表示范围 `0—255`。若用于补码整数，则范围是 `-128—127`。位宽是硬件接口的一部分，输入、输出和 ALU 通路必须保持一致。

## 5. 构建 4 位寄存器

4 位寄存器的结构与 8 位寄存器相同，只是并行存储单元数量减半。它适合保存较短的控制字段、地址的一部分或教学 CPU 中的辅助状态。

```text
D[3:0] ──> [bit3][bit2][bit1][bit0] ──> Q[3:0]
                         ↑
                    CLK / LOAD
```

### 5.1 为什么同时使用不同位宽

CPU 内部不一定所有数据都同宽：

- 8 位寄存器可保存数据和运算结果；
- 4 位寄存器可保存较短的操作码、控制字段或地址字段；
- 不同位宽之间连接时，需要明确高位补 0、截断或分段处理规则。

不能把“4 位寄存器”简单理解为性能更低的 8 位寄存器；它的位宽是由所保存的信息和数据通路设计决定的。

## 6. ALU 加法器接入 CPU

ALU 需要接收操作数、运算控制信号并输出结果。以加法为例：

```text
A 寄存器 ──> ALU A 输入
B 寄存器 ──> ALU B 输入
                    │
             ADD 控制信号
                    ▼
                 ALU 结果
```

### 6.1 加法路径

1. 把第一个操作数装入 ALU 的 A 输入寄存器或对应通路；
2. 把第二个操作数装入 B 输入通路；
3. 控制 ALU 执行加法；
4. 在规定时刻把结果送到总线；
5. 目标寄存器加载总线上的结果。

ALU 本身负责组合运算，寄存器负责在时钟边沿保存结果。这个分工让连续的运算步骤能够逐拍进行。

## 7. CPU 内部原理图中的 ALU 部分

在 CPU 原理图中，ALU 周围通常能看到：

- 两路操作数输入；
- 加法、减法、按位逻辑等运算选择线；
- 结果输出线；
- 进位、溢出或零标志等状态输出；
- 与内部总线相连的输入/输出控制。

### 7.1 从模块图读电路

阅读原理图时可以按下面顺序：

1. 先找数据输入和结果输出；
2. 再找每个输入来自哪个寄存器或总线；
3. 找出决定运算类型的控制信号；
4. 确认结果何时被哪个寄存器接收；
5. 单独追踪进位、溢出和零结果等状态线。

这样可以把一张复杂图拆成“数据流”和“控制流”两部分，而不是试图一次看懂所有连线。

## 8. 内部数据总线

内部数据总线是一组共享的并行线路，用于在寄存器、ALU 和其他 CPU 部件之间传输数据。8 位 CPU 通常使用 8 根数据线表示 `D[7:0]`。

### 8.1 总线的两个基本动作

- **驱动总线：**某个源寄存器通过输出使能，把自己的值放到总线上；
- **加载总线：**某个目标寄存器在时钟边沿采样总线并保存。

抽象写法：

```text
R1 --OE_R1--> BUS --LOAD_R2--> R2
```

### 8.2 总线争用

同一时刻只能有一个输出端主动驱动共享总线。若 `R1` 输出 `0`、`R2` 输出 `1`，两者同时连接总线就会产生冲突。设计控制信号时，要保证未被选中的输出处于高阻态。

## 9. 通过总线完成一次寄存器传送

以 `R1 → R2` 为例，一次典型的同步传送可以拆成两个阶段：

### 阶段一：源寄存器输出

- `OE_R1=1`，允许 R1 驱动总线；
- `OE` 其他寄存器全部无效；
- 总线上的每一位稳定为 R1 的对应位。

### 阶段二：目标寄存器加载

- `LOAD_R2=1`；
- 在规定的时钟边沿，R2 采样总线；
- 时钟边沿结束后，R2 保存 R1 的值；
- 撤销 `OE_R1`。

如果目标寄存器在总线稳定前就加载，或者源寄存器太早断开，都会导致数据传输错误。教学电路里可以把每个控制信号和时钟边沿标在时序表上。

## 10. ALU 结果写回寄存器

把寄存器传送扩展到运算，就得到一条完整的数据通路：

```text
R1 ──> ALU 输入 A
R2 ──> ALU 输入 B
          │
       运算选择
          ▼
       ALU 结果 ──> BUS ──> R3
```

完成 `R3 = R1 + R2` 可以按以下步骤理解：

1. R1 和 R2 分别向 ALU 的两个输入提供操作数；
2. 控制单元把 ALU 设置为 ADD；
3. ALU 组合地产生加法结果；
4. 允许 ALU 结果驱动内部总线；
5. 在时钟边沿使 R3 加载总线数据。

这里的 R3 可以是普通寄存器，也可以是专门保存累加结果的寄存器。关键是结果必须在某个时钟边沿被保存，才能用于下一步。

## 11. CPU 内部结构的完整连接

课程视频 59 将主要模块连接成一个教学 CPU。可用下面的抽象图理解：

```text
             ┌──────────────┐
寄存器组 ───>│              │───> 状态/标志
             │     ALU      │
寄存器组 <───│              │
      ╲      └──────┬───────┘
       ╲            │
        ╲      内部数据总线
         ╲          │
          └── 控制单元/时钟
```

完整连接至少要解决三个问题：

1. 数据从哪个模块出发；
2. 哪个模块在什么时刻接收数据；
3. 控制信号如何保证不会发生总线争用和错误写入。

当模块越来越多时，清晰的命名和模块边界比“把线接通”更重要。

## 12. 控制信号的分类

CPU 内部控制信号可以按作用分成几类：

| 类别 | 示例 | 作用 |
|---|---|---|
| 输出使能 | `OE_R1`、`OE_ALU` | 允许某个模块驱动总线 |
| 加载控制 | `LOAD_R2` | 让目标寄存器在时钟边沿保存数据 |
| 运算选择 | `ADD`、`SUB`、`AND` | 选择 ALU 的操作类型 |
| 复位控制 | `RESET` | 把状态恢复到初始值 |
| 时钟 | `CLK` | 统一规定状态更新时刻 |
| 选择控制 | `SEL` | 在多路输入或多路结果中选择一路 |

控制信号通常不是随意同时变化的。一个可执行的 CPU 步骤，应明确哪些信号在当前周期有效，哪些必须保持关闭。

## 13. 用时钟周期描述一条运算

为了避免“所有事情同时发生”的混乱，可以把运算拆成时钟周期。以 `R3=R1+R2` 为例：

| 周期 | 主要动作 | 控制信号示意 |
|---|---|---|
| T1 | R1、R2 提供 ALU 输入 | 输入选择有效，ALU=`ADD` |
| T2 | ALU 结果驱动总线 | `OE_ALU=1` |
| T3 | R3 保存结果 | `LOAD_R3=1`，在边沿采样 |

不同教学 CPU 可能把这些动作压缩到不同数量的周期，具体取决于 ALU 输入是否有专用寄存器、总线是否单一以及控制器如何设计。重要的是：**数据必须在稳定后采样，状态必须在时钟边沿更新。**

## 14. 数据通路和控制通路的分工

CPU 可以分成两条相互配合的路径：

- **数据通路：**寄存器、ALU、总线以及它们之间的数据线，负责搬运和处理数据；
- **控制通路：**控制单元、时钟和各类使能/选择信号，负责安排动作顺序。

数据通路没有控制信号就不知道什么时候输出、什么时候写入；控制通路没有数据通路就只能产生信号而不能完成运算。

这是一种很有用的分析方法：看到输出结果错误时，先判断是数据本身错误，还是控制时序让正确数据在错误时刻被读写。

## 15. 计算机概念梳理

课程视频 60 对前面内容进行串联，可以用下面的层次回顾：

```text
晶体管/继电器
    ↓
逻辑门（AND、OR、NOT、XOR）
    ↓
加法器、比较器、选择器
    ↓
锁存器、寄存器、内存
    ↓
ALU + 寄存器 + 总线 + 控制信号
    ↓
CPU
    ↓
CPU + 内存 + 输入输出 = 计算机系统
```

每一层都不是凭空出现的：上层组件由下层模块组合而成，并通过更清晰的接口隐藏内部细节。掌握这种层次关系，比孤立记忆某个电路图更重要。

## 16. 搭建和仿真的排错清单

### 数据宽度

- 8 位输入是否接到了 8 位 ALU 端口；
- 4 位寄存器是否发生了意外截断或高位补齐；
- 总线的每一位是否按同一编号连接。

### 控制时序

- 是否存在两个模块同时驱动总线；
- 目标寄存器是否在数据稳定后才加载；
- `LOAD`、`OE` 和时钟边沿是否匹配；
- ALU 运算选择是否在采样前保持稳定。

### 结果验证

- 先单独测试寄存器的写入和保持；
- 再测试寄存器到寄存器的总线传送；
- 最后测试 ALU 结果写回；
- 同时检查普通结果、进位、溢出和复位状态。

不要一开始就运行完整 CPU。按模块、按总线传送、按时钟周期逐层验证，定位问题会更快。

## 17. 一页复习表

| 知识点 | 关键结论 |
|---|---|
| CPU 核心 | 寄存器 + ALU + 总线 + 控制信号 |
| 数据通路 | 数据在寄存器、ALU 和总线之间经过的路径 |
| 8 位寄存器 | 并行保存 8 位数据，受时钟和 LOAD 控制 |
| 4 位寄存器 | 保存较短的控制/地址/字段信息 |
| ALU | 对操作数执行算术和逻辑运算 |
| OE | 允许模块驱动总线，不允许时进入高阻态 |
| LOAD | 让目标寄存器在时钟边沿保存总线数据 |
| 总线 | 多个模块共享的并行数据线路 |
| 写回 | ALU 结果通过总线进入目标寄存器 |
| 控制通路 | 决定谁输出、谁加载、做哪种运算以及何时更新 |

## 18. 自测题与参考答案

1. CPU 内部数据通路包含哪些主要模块？
2. 8 位寄存器的 `LOAD` 和 `OE` 分别做什么？
3. 为什么内部总线同一时刻通常只能有一个主动输出者？
4. 如何完成 `R3=R1+R2` 的数据传送和写回？
5. 4 位寄存器和 8 位寄存器的主要区别是什么？
6. 数据通路和控制通路分别负责什么？
7. 如果 ALU 结果正确但 R3 没有变化，应优先检查哪些信号？
8. 为什么要按时钟周期描述 CPU 内部操作？

### 参考答案

1. 寄存器、ALU、内部数据总线以及产生时序和选择信号的控制单元。
2. `LOAD` 决定是否在时钟边沿保存输入；`OE` 决定是否把已保存的值驱动到总线。
3. 多个输出者同时驱动不同逻辑值会产生总线争用，导致错误甚至硬件损坏。
4. 让 R1、R2 提供 ALU 输入，选择 ADD，使 ALU 输出到总线，再在时钟边沿让 R3 加载总线。
5. 并行存储位数不同，接口位宽和可表示范围不同；结构原理相同。
6. 数据通路搬运和处理数据，控制通路安排输出、加载、运算选择和更新时序。
7. 检查 `OE_ALU`、`LOAD_R3`、时钟边沿、总线连接和 R3 是否处于复位状态。
8. 因为寄存器只在特定时刻更新，周期表能明确数据稳定、传输和写回的先后关系。

## 19. 与原课程资料对应关系

| 本篇内容 | 视频 | 主题 |
|---|---|---|
| 计算机系统核心组件 | 53 | CPU、ALU、寄存器、内存和总线 |
| 8 位寄存器 | 54 | 8 位数据的保存和接口 |
| 4 位寄存器 | 55 | 较窄位宽寄存器的构建 |
| ALU 加法器 | 56 | ALU 与寄存器数据通路 |
| CPU 原理图的 ALU 部分 | 57 | ALU 输入、输出和控制信号 |
| 内部数据总线 | 58 | 总线驱动、加载和连接 |
| CPU 内部结构完成 | 59 | 寄存器、ALU、总线和控制的组合 |
| 计算机概念梳理 | 60 | 从逻辑门到计算机系统的层次关系 |

**资料来源：**《自己动手做台计算机》课程，百度网盘分享链接：`https://pan.baidu.com/s/1RLiC8gC1sXTu7EgkQs9oHw?pwd=6633`，提取码：`6633`。本篇按视频 53—60 整理；第 61 个视频起的汇编、指令集和手动执行程序放入第 06 篇。